# Sparse formats

A sparse matrix is stored as its nonzeros and their positions, and the format
decides how the positions are written down. MTL5 has three:

- **COO**, coordinate: one (row, column, value) triplet per entry, in any order.
  It is the format to assemble in.
- **CSR**, compressed rows: the entries row by row, so the row indices collapse
  to one offset per row. It is the format to compute with.
- **ELL**, ELLPACK: every row padded to the same width. Its regularity suits
  vector units and GPUs, and it pays for that in padding.

This notebook builds the C++ example's 2-D Laplacian in all three, and counts
what each stores. Then it tries ELL on a circuit matrix, whose rows are nothing
like uniform. Last, it writes the matrices to Matrix Market files and reads them
back.

**Uses:** `mtl5.sparse.coo_matrix` with `.insert`, `.sort` and `.tocsr`,
`mtl5.sparse.csr_matrix`, `mtl5.sparse.ell_matrix`,
`mtl5.generators.laplacian_2d`, `mtl5.io.mm_write_sparse`, `mtl5.io.mm_read`,
`mtl5.io.mm_write` and `mtl5.io.mm_read_dense`.

**From C++:** [`phase07_sparse_formats/sparse_formats.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase07_sparse_formats/sparse_formats.cpp).

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

BYTES = 8  # a float64 value, and an int64 index

## COO: assemble entry by entry

The matrix is the 5-point Laplacian on a 6 × 6 grid of unknowns, numbered row by
row: 4 on the diagonal, −1 for each of the up to four neighbours. Assembly visits
each grid point and appends its entries. In COO that is all `insert` does: it
appends a triplet. A repeated (row, column) adds to the earlier one when the
matrix is read, rather than replacing it, which is what finite element assembly
needs (see [FEM assembly and reordering](fem_assembly.ipynb)). `sort` puts
the triplets in row-major order.

In [ ]:
grid = 6
N = grid * grid

empty = np.array([], dtype=np.int64)
coo = ms.coo_matrix(empty, empty, np.array([]), (N, N))
for iy in range(grid):
    for ix in range(grid):
        row = ix + iy * grid
        coo.insert(row, row, 4.0)
        if ix > 0:
            coo.insert(row, row - 1, -1.0)
        if ix + 1 < grid:
            coo.insert(row, row + 1, -1.0)
        if iy > 0:
            coo.insert(row, row - grid, -1.0)
        if iy + 1 < grid:
            coo.insert(row, row + grid, -1.0)

nnz = coo.nnz
print(f"{N} unknowns, {nnz} triplets, sorted: {coo.is_sorted}")
# Each row's diagonal went in before its left neighbour, so the order is off.
assert not coo.is_sorted
coo.sort()
print("after sort():", coo.is_sorted)

# 4 corners with 3 entries, 4 (grid - 2) edge points with 4, and the rest with 5.
assert nnz == 5 * N - 4 * grid == 156
assert coo.is_sorted

## CSR: compress the rows

`tocsr` sorts the triplets, sums any duplicates, and replaces the row of every
entry with one offset per row: `indptr[i]` is where row i starts, and
`indptr[i + 1] − indptr[i]` is its length. Column indices and values stay one
per entry. The result is the same matrix that `mtl5.generators.laplacian_2d`
builds.

In [ ]:
csr = coo.tocsr()
indptr, indices, data = csr.to_csr_arrays()
print("indptr[:8]  =", indptr[:8])
print("row 0:   columns", indices[indptr[0] : indptr[1]], "values", data[indptr[0] : indptr[1]])
print("row 7:   columns", indices[indptr[7] : indptr[8]], "values", data[indptr[7] : indptr[8]])

reference = mtl5.generators.laplacian_2d(grid, grid)
for ours, theirs in zip(csr.to_csr_arrays(), reference.to_csr_arrays()):
    assert np.array_equal(ours, theirs)
assert csr.nnz == nnz and len(indptr) == N + 1

# The same three arrays build a CSR matrix directly, with no COO step.
again = ms.csr_matrix(data, indices, indptr, (N, N))
assert np.array_equal(again.to_csr_arrays()[2], data)

## ELL: pad every row to the widest

ELL stores an N × w block of column indices and another of values, where w is
the widest row's length. Shorter rows are padded: their index is −1 and their
value 0. Here the widest row has 5 entries and the corner rows 3, so ELL holds
24 empty slots out of 180.

In [ ]:
ell = ms.ell_matrix(csr)
ell_idx, ell_val = ell.to_ell_arrays()
print(f"max_width {ell.max_width}, {ell.stored} slots for {ell.nnz} entries,", end=" ")
print(f"padding {ell.padding_ratio:.1%}")
print("row 0 (a corner):", ell_idx[0], ell_val[0])
print("row 7 (interior):", ell_idx[7], ell_val[7])

assert ell.max_width == 5
assert ell.stored == N * ell.max_width == 180
assert ell.stored - ell.nnz == 24
assert (ell_idx == -1).sum() == 24

## What each format stores

Counting the 8-byte values and indices each format holds:

- **Dense** holds all N² values.
- **COO** holds 3 per entry.
- **CSR** holds 2 per entry, plus N + 1 row offsets.
- **ELL** holds 2 per slot, padding included.

At 36 unknowns, the sparse formats already use about a quarter of the dense
storage. The difference grows with the grid: dense storage grows as N², the
sparse ones as N. On the small grid, CSR is the smallest. On a large grid ELL
is, slightly: the Laplacian pads only its boundary rows, which costs less
than CSR's offset for every row.

In [ ]:
def storage(A):
    # (values, index entries) each layout stores for the sparse matrix A.
    n, nz, w = A.shape[0], A.nnz, ms.ell_matrix(A).max_width
    return {
        "dense": (n * n, 0),
        "COO": (nz, 2 * nz),
        "CSR": (nz, nz + n + 1),
        "ELL": (n * w, n * w),
    }


table = storage(csr)
print(f"{'format':>6} {'values':>7} {'indices':>8} {'total':>6} {'bytes':>7} {'vs dense':>9}")
for name, (vals, idx) in table.items():
    total = vals + idx
    print(f"{name:>6} {vals:7} {idx:8} {total:6} {total * BYTES:7} {total / (N * N):9.1%}")

assert {k: sum(v) for k, v in table.items()} == {"dense": 1296, "COO": 468, "CSR": 349, "ELL": 360}

grids = np.array([6, 10, 20, 40, 80, 160])
sizes = {name: [] for name in table}
for g in grids:
    for name, (vals, idx) in storage(mtl5.generators.laplacian_2d(int(g), int(g))).items():
        sizes[name].append((vals + idx) * BYTES)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for (name, ys), color in zip(sizes.items(), ["#8a8984", *SERIES]):
    ax.loglog(grids**2, ys, "o-", color=color, label=name, markersize=4)
ax.set_xlabel("unknowns N")
ax.set_ylabel("bytes")
ax.set_title("Storage of the 5-point Laplacian")
ax.grid(True, which="major")
ax.legend(frameon=False)
plt.show()

# On a large grid ELL is the smallest: 10 values per unknown against CSR's 11,
# because only the boundary rows are padded, and CSR pays one offset per row.
last = {name: ys[-1] for name, ys in sizes.items()}
assert last["ELL"] < last["CSR"] < last["COO"] < last["dense"] / 1000
assert sum(table["CSR"]) < sum(table["ELL"])  # at 6 x 6, 20 of the 36 rows are padded

## ELL on a matrix with irregular rows

The Laplacian is the case ELL is built for: every row has nearly the same
length. A circuit matrix is the opposite. Most nodes connect to a few devices,
but a supply or ground net connects to many. `adder_dcop_01`, from Sandia's
circuit simulations (in `notebooks/data`, see
[Circuit matrices with KLU](circuit_klu.ipynb)), has a median row length of 6
and one row of 1310. ELL pads every row to 1310.

In [ ]:
A = mtl5.io.mm_read("../data/adder_dcop_01.mtx")
n = A.shape[0]
lengths = np.diff(A.to_csr_arrays()[0])
E = ms.ell_matrix(A)
csr_bytes = sum(storage(A)["CSR"]) * BYTES
ell_bytes = 2 * E.stored * BYTES
print(
    f"n = {n}, nnz = {A.nnz:,}, row lengths: median {np.median(lengths):.0f}, max {lengths.max()}"
)
print(f"ELL: {E.stored:,} slots, {E.padding_ratio:.1%} padding, {ell_bytes / 2**20:.1f} MiB")
print(f"CSR: {csr_bytes / 2**10:.0f} KiB; ELL is {ell_bytes / csr_bytes:.0f} times larger")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 30), color=SERIES[0])
ax.axvline(E.max_width, color=SERIES[1], linewidth=1.5)
ax.text(E.max_width, ax.get_ylim()[1] * 0.9, "  ELL width", color=SERIES[1], ha="right", va="top")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("entries in the row")
ax.set_ylabel("rows")
ax.set_title("Row lengths of adder_dcop_01")
plt.show()

assert E.max_width == lengths.max() == 1310
assert E.padding_ratio > 0.99
assert ell_bytes > 100 * csr_bytes

## The formats agree

All three compute the same product as the dense matrix, to rounding: with a
random x, the only difference is the order in which each row's terms are added.

In [ ]:
x = np.random.default_rng(147).standard_normal(N)
dense = np.zeros((N, N))
rows_, cols_, vals_ = coo.to_coo_arrays()
np.add.at(dense, (rows_, cols_), vals_)

y_dense = dense @ x
for name, M in (("COO", coo), ("CSR", csr), ("ELL", ell)):
    y = np.asarray(M.matvec(x))
    err = np.abs(y - y_dense).max()
    print(f"{name}: max |y - y_dense| = {err:.1e}")
    assert err <= 1e-14

## Matrix Market

Matrix Market is the text format of the NIST Matrix Market and the SuiteSparse
Matrix Collection. A sparse matrix is written as `coordinate`, one entry per
line with 1-based indices. A dense one is written as `array`, column by column.
MTL5 writes values with 17 significant digits, enough for a float64 to read
back to the same bits.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    sparse_file = os.path.join(tmp, "laplacian.mtx")
    mtl5.io.mm_write_sparse(sparse_file, csr, "2D Laplacian 6x6 grid")
    with open(sparse_file, encoding="utf-8") as f:
        head = [next(f).rstrip() for _ in range(6)]
    back = mtl5.io.mm_read(sparse_file)

    block = dense[:4, :4] / 3  # thirds, which no short decimal represents
    dense_file = os.path.join(tmp, "block.mtx")
    mtl5.io.mm_write(dense_file, block, "top-left 4x4 block, divided by 3")
    block_back = mtl5.io.mm_read_dense(dense_file).to_numpy()

print("\n".join(head))
for ours, theirs in zip(back.to_csr_arrays(), csr.to_csr_arrays()):
    assert np.array_equal(ours, theirs)
assert np.array_equal(block_back, block)  # bit for bit
print("sparse and dense round trips: exact")

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `mat::coordinate2D<double> coo(N, N); coo.insert(i, j, v)` | `coo = ms.coo_matrix(row, col, data, (N, N))`, then `coo.insert(i, j, v)` |
| `coo.sort(); coo.compress()` | `coo.sort()`; `coo.tocsr()`, which also sorts |
| `mat::inserter<compressed2D<double>> ins(crs, 5); ins[i][j] << v` | assemble in COO, or `ms.csr_matrix(data, indices, indptr, shape)` from arrays |
| `mat::ell_matrix<double> ell(crs)`, `ell.max_width()` | `ms.ell_matrix(csr)`, `.max_width`, `.stored`, `.padding_ratio` |
| `io::mm_write_sparse(file, A, comment)`, `io::mm_read<double>(file)` | `mtl5.io.mm_write_sparse(path, A, comment)`, `mtl5.io.mm_read(path)` |
| `io::mm_write(file, D, comment)`, `io::mm_read_dense<double>(file)` | `mtl5.io.mm_write(path, D, comment)`, `mtl5.io.mm_read_dense(path)` |

The C++ program counts storage in values and compares entries one at a time.
Here the counts are asserted, and the Matrix Market round trips are checked
bit for bit. The circuit matrix is added, to show the case where ELL's
padding outweighs its regularity.